In [0]:
import requests
import spark
def stop_interpreter():
    notebook_url = "http://your_server"
    notebook_port = 8998
    notebook_id = z.getInterpreterContext().getNoteId()
    endpoint = f"{notebook_url}:{notebook_port}/api/interpreter/setting/restart/spark"
    headers = {
        "Content-Type": "application/json"
    }
    payload = {
        "noteId": notebook_id  # Please replace this with your actual Note ID
    }
    print(f"requests",endpoint,payload,headers)
    response = requests.put(endpoint, json=payload, headers=headers)
    print("Status Code:", response.status_code)
    print("Response Body:", response.text)

In [ ]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark import StorageLevel

In [ ]:
appName = "Database_TableName"
tableName = f"delta_tableName"

In [ ]:
config_row = spark.sql(f"""
    SELECT connection_string, username, password, driver
    FROM connection_db_config      
    WHERE is_active = true
    AND connection_name = 'your_connection_name'
    """).first()
if not config_row:
    print("Connection config not found")
    stop_interpreter()
connection_string = config_row['connection_string']
username = config_row['username'] 
password = config_row['password']
driver = config_row['driver']

In [ ]:
df = spark.read\
    .format("jdbc") \
    .option("url", connection_string) \
    .option("dbtable", f"""(SELECT * FROM tableName)""")\
    .option("user", username) \
    .option("password", password) \
    .option("driver", driver) \
    .load()
# Cache ไว้เพื่อใช้ในการ MERGE 2 ครั้ง
df.persist(StorageLevel.MEMORY_AND_DISK)
print(f"Start create df to Temp...")

# to store data for temporary 
df.createOrReplaceTempView("tableName_TEMP_VIEW")
print(f"Task create df to Temp is completed.")
print(f"Total rows loaded: {df.count()}")

In [ ]:
transform_df = spark.sql(f"""SELECT * FROM tableName_TEMP_VIEW""")
#transform_df = transform_df.toDF(*[c.lower() for c in transform_df.columns])
transform_df = transform_df.withColumn(
    "partition_date",
    date_format(to_date(col("timestamp"), "yyyy-MM-dd"), "yyyyMMdd").cast("long"))
transform_df.createOrReplaceTempView("tableName_TEMP_VIEW_FINAL")

# PARTITION_DATE is the column used for partitioning the Delta table. You can change it to the appropriate column in your case.

In [ ]:
#For update
print(f"Start update data...")
spark.sql(f"""
    MERGE INTO {tableName} t
    USING (SELECT * FROM tableName_TEMP_VIEW_FINAL) s
    ON  t.col1 = s.col1 and                     
        t.col2 = s.col2                         # col1 and col2 are the primary keys of the table

    WHEN MATCHED THEN
        UPDATE SET
            col3 = s.col3,
            col4 = s.col4,
            col5 = s.col5

    
    WHEN NOT MATCHED
        THEN INSERT (col1, col2, col3, col4, col5, PARTITION_DATE)
        VALUES (
                s.col1,
                s.col2,
                s.col3,
                s.col4,
                s.col5,
                s.PARTITION_DATE)
""")
print(f"Task insert data is completed.")

In [ ]:
stop_interpreter()